Setup

In [4]:
!pip install transformers torch -q

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)  # should print "cuda" — if not, go to Runtime > Change runtime type > GPU

cuda


In [5]:
target_name = "gpt2-medium"
draft_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(target_name)
target_model = AutoModelForCausalLM.from_pretrained(target_name).to(device)
draft_model = AutoModelForCausalLM.from_pretrained(draft_name).to(device)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Decode (no cache)

In [24]:
import time
prompt = "The weather today is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated = input_ids
num_new_tokens = 500

start = time.time()
for step in range(num_new_tokens):
    with torch.no_grad():
        outputs = target_model(generated)
    next_token_logits = outputs.logits[0, -1, :]

    next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)
    generated = torch.cat([generated, next_token_id], dim=1)
end = time.time()


print(tokenizer.decode(generated[0]))
print("Time taken: ", end - start)

The weather today is very cold and wet. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out

Decode (with KV cache)

In [26]:
prompt = "The weather today is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

past_key_values = None
current_token = input_ids
generated = input_ids
num_new_tokens = 500

start = time.time()
for step in range(num_new_tokens):
    with torch.no_grad():
        outputs = target_model(current_token, past_key_values=past_key_values, use_cache=True)

    next_token_logits = outputs.logits[0, -1, :]
    next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)

    generated = torch.cat([generated, next_token_id], dim=1)
    past_key_values = outputs.past_key_values
    current_token = next_token_id
end = time.time()

print(tokenizer.decode(generated[0]))
print("Time taken (cached): ", end - start)

The weather today is very cold and wet. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out of the house for a while. I am going to be out

Full Speculative Decoding Script

In [11]:
import time

prompt = input("Enter your prompt: ")
num_tokens_wanted = int(input("Enter the number of tokens you want to generate: "))

num_tokens_generated = 0
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
generated = input_ids

total_accepted = 0
total_rounds = 0
start_time = time.time()

while num_tokens_generated < num_tokens_wanted:
  n = generated.shape[1]

  past_key_values = None
  current_token = generated
  num_new_tokens = 5
  candidate_tokens = []
  candidate_probs = []

  for step in range(num_new_tokens):
    with torch.no_grad():
      outputs = draft_model(current_token, past_key_values=past_key_values, use_cache=True)
      next_token_logits = outputs.logits[0, -1, :]

      probs = torch.softmax(next_token_logits, dim=-1)
      candidate_probs.append(probs)

      next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)
      candidate_tokens.append(next_token_id)
      past_key_values = outputs.past_key_values
      current_token = next_token_id

  verify_input = torch.cat([generated] + candidate_tokens, dim=1)
  outputs = target_model(verify_input)
  token_logits = outputs.logits[0, n-1:n-1+num_new_tokens, :]
  probs = torch.softmax(token_logits, dim=-1)
  num_accepted = 0
  rejected = False

  for i in range(len(candidate_tokens)):
    token_id = candidate_tokens[i].item()
    q_x = candidate_probs[i][token_id]
    p_x = probs[i][token_id]

    accept_prob = min(1.0, p_x/q_x)
    if torch.rand(1).item() < accept_prob:
      num_accepted += 1
      continue
    else:
      adjusted = torch.clamp(probs[i] - candidate_probs[i], 0)
      adjusted = adjusted/adjusted.sum()
      resampled_token_id = torch.multinomial(adjusted, num_samples=1).unsqueeze(0)
      rejected = True
      break

  if rejected:
    generated = torch.cat([generated] + candidate_tokens[:num_accepted] + [resampled_token_id], dim=1)
  else:
    generated = torch.cat([generated] + candidate_tokens, dim=1)
    bonus_token_logits = outputs.logits[0, n-1+num_new_tokens, :]
    bonus_token_id = torch.argmax(bonus_token_logits).unsqueeze(0).unsqueeze(0)
    generated = torch.cat([generated, bonus_token_id], dim=1)

  num_tokens_generated = generated.shape[1] - input_ids.shape[1]
  total_accepted += num_accepted
  total_rounds += 1

elapsed = time.time() - start_time

print(tokenizer.decode(generated[0]))
print(f"Tokens generated: {num_tokens_generated}")
print(f"Rounds: {total_rounds}")
print(f"Avg accepted per round: {total_accepted / total_rounds:.2f} / {num_new_tokens}")
print(f"Time: {elapsed:.2f}s")
print(f"Tokens/sec: {num_tokens_generated / elapsed:.2f}")

Enter your prompt: The weather today is
Enter the number of tokens you want to generate: 500
The weather today is very good, and we are going to be able to get some good results tomorrow.

"We want to get the best out of the guys, and we want to get a lot started. We know the intensity of this game, and we want to get the pressure off the guys.

"We don't want to be sitting back and waiting for the game to finish. We want to get the boys ready for the game, and make sure we form a good team.

"We want to get the boys flying, and start the stronger. All the guys are ready to go, and we want to get the boys ready for the game."<|endoftext|>The U.S. Department of Justice has filed a lawsuit against the company that owns the National Security Agency's (NSA) Today email system, alleging that the company violated the Foreign Intelligence Surveillance Act (FISA) by collecting bulk data on Americans.

The lawsuit, filed in federal court in Washington, D.C., alleges that the company, Equifax, v